# The COMPAS Controversy: When Two Kinds of Fairness Can't Both Hold

In this notebook I look at the 2016 controversy over COMPAS, a risk-assessment tool used in the US criminal justice system, as a case study in something sharper than "the algorithm was biased": two sides ran the same numbers, both found a real and correctly measured disparity, and both were right, because they were measuring fairness in two different, and it turns out mathematically incompatible, ways.

This is the natural next step after the Tokyo Medical University notebook. That case was about a single, deliberate, group-level adjustment. This one is about what happens even without one, when a score is honestly calibrated and the two groups it is scoring simply have different underlying rates of the thing being predicted.

In this notebook, I will:

- Summarize what ProPublica and Northpointe each reported, and where they actually disagreed
- Build a small synthetic simulation of two groups with different base rates, scored by a score that is honestly calibrated
- Measure error rates the way ProPublica did, and calibration the way Northpointe did, on the same simulated data
- Try to fix the error-rate gap with a group-specific threshold, and see what it does to calibration
- Write down what this implies for any AI system that scores people

## 1. Background: What Was Reported

COMPAS is a risk-assessment tool that produces a score meant to predict how likely a defendant is to reoffend, used in parts of the US criminal justice system to help inform decisions like bail and sentencing.

In 2016, ProPublica published an investigation into COMPAS scores for thousands of defendants and reported that Black defendants who did not go on to reoffend were substantially more likely than white defendants who did not reoffend to have been labeled high risk, a higher false positive rate. It also reported the mirror problem: white defendants who did reoffend were more likely than Black defendants who reoffended to have been labeled low risk, a higher false negative rate.

Northpointe, the company behind COMPAS, disputed that framing. It pointed out that COMPAS was well calibrated: among defendants who received the same score, the actual reoffense rate was similar regardless of race.

Both claims held up under scrutiny. What followed was a body of academic work showing that this was not a case of one side making a measurement error. When two groups have different underlying base rates for the outcome being predicted, it is mathematically impossible for a score to be simultaneously well calibrated and have equal false positive and false negative rates across those groups, outside of special cases like a perfect predictor. ProPublica and Northpointe were each measuring a real and legitimate notion of fairness, and the two cannot both hold at once.

## 2. A Question This Notebook Does Not Answer

Everything above treats "reoffends" as a clean, measurable fact, but the real data behind COMPAS measures rearrest, not some ground truth of who actually committed another crime. Rearrest rates are shaped by where and how policing happens, not only by behavior, which was itself part of the controversy.

This notebook does not weigh in on that question, and does not try to reproduce the real COMPAS dataset or the real demographic groups involved. What it builds instead is a synthetic simulation using two unlabeled groups, Group A and Group B, to isolate a narrower, purely mathematical question: given that two groups have different base rates, for whatever reason, what happens to a fairness check built on error rates versus one built on calibration? That question has a clean, demonstrable answer, independent of what caused the base rates to differ in the first place.

## 3. Simulating Two Groups With Different Base Rates

Each simulated person gets a true underlying probability of reoffending, drawn from a beta distribution, then an actual outcome drawn from that probability. The score I will use later is that same true probability, so by construction the score is perfectly calibrated, any calibration gap that shows up later is not coming from a flawed model, it is coming from something else entirely.

In [ ]:
import random

random.seed(7)


def simulate_population(count, alpha, beta):
    """Returns a list of (score, reoffended) pairs from a beta-distributed risk."""
    people = []
    for _ in range(count):
        true_risk = random.betavariate(alpha, beta)
        reoffended = random.random() < true_risk
        people.append((true_risk, reoffended))
    return people


group_a = simulate_population(4000, alpha=2, beta=5)
group_b = simulate_population(4000, alpha=4, beta=3)

## 4. Checking the Base Rates

I picked different shape parameters for the two groups on purpose, so I check that this actually produced two different overall reoffense rates before going any further.

In [ ]:
def base_rate(population):
    """Returns the fraction of a population that reoffended."""
    reoffended = [r for _, r in population if r]
    return len(reoffended) / len(population)


print("Group A base rate:", round(base_rate(group_a), 3))
print("Group B base rate:", round(base_rate(group_b), 3))

## 5. Measuring Error Rates, the ProPublica Way

ProPublica's analysis centered on two rates: among people who did *not* reoffend, what fraction were labeled high risk anyway, the false positive rate, and among people who *did* reoffend, what fraction were labeled low risk, the false negative rate. I apply one shared threshold to both groups, the same score cutoff for everyone, which is the fairest-sounding starting point.

In [ ]:
THRESHOLD = 0.5


def false_positive_rate(population, threshold=THRESHOLD):
    """Returns the fraction of non-reoffenders scored at or above the threshold."""
    non_reoffenders = [score for score, r in population if not r]
    flagged = [score for score in non_reoffenders if score >= threshold]
    return len(flagged) / len(non_reoffenders)


def false_negative_rate(population, threshold=THRESHOLD):
    """Returns the fraction of reoffenders scored below the threshold."""
    reoffenders = [score for score, r in population if r]
    missed = [score for score in reoffenders if score < threshold]
    return len(missed) / len(reoffenders)

## 6. Testing the Error Rates at a Shared Threshold

I compute both rates for each group, using the exact same threshold for both, and compare.

In [ ]:
print("Group A false positive rate:", round(false_positive_rate(group_a), 3))
print("Group B false positive rate:", round(false_positive_rate(group_b), 3))
print("Group A false negative rate:", round(false_negative_rate(group_a), 3))
print("Group B false negative rate:", round(false_negative_rate(group_b), 3))

## 7. Measuring Calibration, the Northpointe Way

Calibration asks a different question: among people who got roughly the same score, did they actually reoffend at roughly the same rate, regardless of group? I bucket people by score into bands and compare the actual reoffense rate within each band, across both groups.

In [ ]:
def calibration_by_band(population, band_width=0.2):
    """Returns, for each score band, the actual reoffense rate within it."""
    bands = {}
    band_count = int(1 / band_width)
    for score, reoffended in population:
        band = min(int(score / band_width), band_count - 1)
        bands.setdefault(band, []).append(reoffended)
    result = {}
    for band, outcomes in bands.items():
        low = round(band * band_width, 1)
        high = round(low + band_width, 1)
        result[(low, high)] = round(sum(outcomes) / len(outcomes), 3)
    return dict(sorted(result.items()))

## 8. Testing Calibration on Both Groups

I print each group's calibration table side by side. Despite the very different error rates measured above, I expect a given score band to correspond to a similar actual reoffense rate in both groups, which is exactly the pattern Northpointe pointed to.

In [ ]:
print("Group A calibration:", calibration_by_band(group_a))
print("Group B calibration:", calibration_by_band(group_b))

## 9. Why Both Sides Were Right

The error rates in step 6 and the calibration tables in step 8 came from the exact same simulated scores, at the exact same threshold, on data where the score is by construction an honest, unbiased estimate of each person's true risk. Nothing about how the score was built favors either group.

The error rates still diverge sharply, because the groups have different base rates, so the same threshold catches a different share of each group's distribution. The calibration tables still line up closely, because the score was never distorted in the first place. Both results are correct simultaneously, which is exactly the finding that made this such a hard case: there is no bug to fix here, and no single number that can confirm the system is fair, because fair according to which definition is doing real work in that sentence.

## 10. Trying to Fix the Error Rate Gap

It is tempting to just raise the threshold for whichever group has the higher false positive rate until the rates match. I search for the threshold on group B that gets its false positive rate as close as possible to group A's false positive rate at the original shared threshold.

In [ ]:
target_fpr = false_positive_rate(group_a, threshold=THRESHOLD)

best_threshold, best_gap = None, None
for candidate in [i / 100 for i in range(100)]:
    gap = abs(false_positive_rate(group_b, threshold=candidate) - target_fpr)
    if best_gap is None or gap < best_gap:
        best_threshold, best_gap = candidate, gap

print("Group A threshold:", THRESHOLD)
print("Group B threshold needed to match group A's false positive rate:", best_threshold)

## 11. Checking What the Fix Did to Calibration

The false positive rates now roughly match, but the two groups are no longer using the same rule to decide who counts as high risk. I check what "high risk" now actually means, in terms of real reoffense rate, for each group under its own threshold.

In [ ]:
def actual_rate_above(population, threshold):
    """Returns the actual reoffense rate among people scored at or above the threshold."""
    flagged = [reoffended for score, reoffended in population if score >= threshold]
    return sum(flagged) / len(flagged)


print(
    "Group A, 'high risk' means an actual reoffend rate of:",
    round(actual_rate_above(group_a, THRESHOLD), 3),
)
print(
    "Group B, 'high risk' means an actual reoffend rate of:",
    round(actual_rate_above(group_b, best_threshold), 3),
)

## 12. What This Result Shows

Matching the false positive rates required moving group B's threshold well above group A's, and once that happens, a "high risk" label no longer points at the same real-world likelihood of reoffending in each group. Fixing the error-rate gap broke calibration in the same motion.

This is not a flaw in this particular simulation, it is the impossibility result itself, made concrete: with two groups at different base rates, a shared rule keeps calibration but produces unequal error rates, and a group-specific rule can equalize an error rate but breaks calibration. There is no threshold policy, shared or group-specific, that gets both at once here.

## 13. Lessons for AI Systems That Score People

What I take from working through this:

- "Is this system fair" is not one question. Calibration and equal error rates are both legitimate, and when base rates differ between groups, a system generally cannot satisfy both, so asking which one matters more for a given decision is unavoidable, not a technicality to route around.
- A vendor's claim that a score is "unbiased" and a critic's claim that its error rates are skewed can both be true at the same time. Neither claim alone settles whether the system should be used the way it is being used.
- Patching one fairness metric by adjusting a threshold is a real, measurable tradeoff against another metric, not a free fix, and that tradeoff should be stated explicitly to whoever is deciding whether to deploy the system, not discovered later.
- Any system I build that scores people for a consequential decision needs an explicit answer to "which fairness definition are we optimizing for, and what does that cost along the other definitions" before it ships, not as an afterthought once a gap is reported.